# 03 - LoRA Fine-Tuning with Qwen2.5

This notebook fine-tunes a pretrained instruction model using LoRA for cardiovascular question answering.



In [1]:
import json
import os
from pathlib import Path

import torch
import pandas as pd

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA")

2.12.0.dev20260408+cu128
True
NVIDIA GeForce RTX 5050 Laptop GPU


In [3]:
BASE_DIR = Path.cwd()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = MODEL_DIR / "qwen2_5_1_5b_cardio_lora"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Base directory:", BASE_DIR)
print("Model:", MODEL_NAME)
print("Output directory:", OUTPUT_DIR)
print("Device:", device)

Base directory: d:\CardioBot_NLP_Final
Model: Qwen/Qwen2.5-1.5B-Instruct
Output directory: d:\CardioBot_NLP_Final\models\qwen2_5_1_5b_cardio_lora
Device: cuda


In [4]:
def read_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data


train_data = read_jsonl(PROCESSED_DIR / "train.jsonl")
val_data = read_jsonl(PROCESSED_DIR / "val.jsonl")

print("Train size:", len(train_data))
print("Validation size:", len(val_data))

pd.DataFrame(train_data).head()

Train size: 101
Validation size: 25


,id,topic,source,question,answer
0,qa_131,Heart Failure,heart_failure.txt,What is heart failure?,"Heart failure, also called congestive heart fa..."
1,qa_143,Prevention,prevention.txt,What is a heart-healthy diet?,"A heart-healthy diet includes vegetables, frui..."
2,qa_075,Cholesterol,Cholesterol.txt,How can saturated and trans fats affect choles...,Foods high in saturated and trans fats can inc...
3,qa_088,Arrhythmia,Arrhythmia.txt,What are the main types of arrhythmia?,Arrhythmias can be classified by where they st...
4,qa_003,Angioplasty and Stent,Angioplasty and stent.txt,When is coronary angioplasty used?,Coronary angioplasty may be used when coronary...


In [5]:
SYSTEM_PROMPT = (
    "You are CardioBot, a helpful cardiovascular health education assistant. "
    "Answer clearly and accurately using simple medical language. "
    "Do not provide diagnosis, prescriptions, or emergency medical decisions. "
    "If the question involves emergency symptoms, advise the user to seek immediate medical help."
)


def format_chat(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]},
    ]
    
    return {"messages": messages}


train_formatted = [format_chat(item) for item in train_data]
val_formatted = [format_chat(item) for item in val_data]

print(train_formatted[0])

{'messages': [{'role': 'system', 'content': 'You are CardioBot, a helpful cardiovascular health education assistant. Answer clearly and accurately using simple medical language. Do not provide diagnosis, prescriptions, or emergency medical decisions. If the question involves emergency symptoms, advise the user to seek immediate medical help.'}, {'role': 'user', 'content': 'What is heart failure?'}, {'role': 'assistant', 'content': 'Heart failure, also called congestive heart failure, is a syndrome in which the heart cannot pump blood effectively because of structural or functional impairment. Common symptoms include shortness of breath, fatigue, and edema.'}]}


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

d:\CardioBot_NLP_Final\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded.
Pad token: <|endoftext|>


In [7]:
def apply_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}


train_texts = [apply_template(item) for item in train_formatted]
val_texts = [apply_template(item) for item in val_formatted]

train_dataset = Dataset.from_list(train_texts)
val_dataset = Dataset.from_list(val_texts)

print(train_dataset[0]["text"])

<|im_start|>system
You are CardioBot, a helpful cardiovascular health education assistant. Answer clearly and accurately using simple medical language. Do not provide diagnosis, prescriptions, or emergency medical decisions. If the question involves emergency symptoms, advise the user to seek immediate medical help.<|im_end|>
<|im_start|>user
What is heart failure?<|im_end|>
<|im_start|>assistant
Heart failure, also called congestive heart failure, is a syndrome in which the heart cannot pump blood effectively because of structural or functional impairment. Common symptoms include shortness of breath, fatigue, and edema.<|im_end|>



In [8]:
def count_tokens(example):
    tokens = tokenizer(example["text"], truncation=False)
    return len(tokens["input_ids"])


train_token_lengths = [count_tokens(item) for item in train_texts]
val_token_lengths = [count_tokens(item) for item in val_texts]

print("Train token length summary:")
print(pd.Series(train_token_lengths).describe())

print("\nValidation token length summary:")
print(pd.Series(val_token_lengths).describe())

Train token length summary:
count    101.000000
mean     115.267327
std        9.359371
min       98.000000
25%      109.000000
50%      114.000000
75%      120.000000
max      145.000000
dtype: float64

Validation token length summary:
count     25.000000
mean     115.080000
std        9.733619
min      102.000000
25%      109.000000
50%      112.000000
75%      122.000000
max      141.000000
dtype: float64


In [9]:
torch_dtype = torch.float16 if device == "cuda" else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch_dtype,
    trust_remote_code=True
)

model.config.use_cache = False

print("Base model loaded.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded.


In [10]:
# setup LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [11]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    fp16=True if device == "cuda" else False,
    use_cpu=True if device == "cpu" else False,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [12]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/101 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/101 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

In [13]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.570354,0.590371,0.568565,11743.000000,0.842522
2,0.553546,0.561771,0.539192,23486.000000,0.850597
3,0.393031,0.567069,0.458668,35229.000000,0.848848


TrainOutput(global_step=78, training_loss=0.7910098540477264, metrics={'train_runtime': 181.0885, 'train_samples_per_second': 1.673, 'train_steps_per_second': 0.431, 'total_flos': 278923421611008.0, 'train_loss': 0.7910098540477264, 'epoch': 3.0})

In [14]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

print("LoRA adapter saved to:", OUTPUT_DIR)

LoRA adapter saved to: d:\CardioBot_NLP_Final\models\qwen2_5_1_5b_cardio_lora


In [15]:
def generate_answer(question, max_new_tokens=160):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)

    return answer.strip()


question = "What is cardiomyopathy?"
answer = generate_answer(question)

print("Question:", question)
print("Answer:", answer)

Question: What is cardiomyopathy?
Answer: Cardiomyopathy means that the heart muscle becomes diseased and cannot pump blood effectively. It can be caused by genetic factors, infections, toxins, alcohol use, high blood pressure, diabetes, obesity, autoimmune diseases, or other conditions.


In [16]:
test_data = read_jsonl(PROCESSED_DIR / "test.jsonl")

finetuned_results = []

for i, item in enumerate(test_data, start=1):
    question = item["question"]
    reference = item["answer"]
    
    pred = generate_answer(question)

    finetuned_results.append({
        "id": item["id"],
        "topic": item["topic"],
        "source": item["source"],
        "question": question,
        "reference_answer": reference,
        "finetuned_answer": pred
    })

    print(f"[{i}/{len(test_data)}] {question}")
    print(pred[:200])
    print("-" * 80)

[1/34] What is dilated cardiomyopathy?
Dilated cardiomyopathy is a type of cardiomyopathy in which the heart muscle becomes enlarged and loses its normal strength and ability to pump blood effectively.
--------------------------------------------------------------------------------
[2/34] What is valve regurgitation?
Valve regurgitation occurs when one of the heart valves does not close properly. Blood leaks backward through the valve instead of flowing in only one direction. This can happen with any heart valve b
--------------------------------------------------------------------------------
[3/34] What are the main functions of blood flow?
Blood flow carries oxygen and nutrients from the lungs and digestive system to tissues throughout the body. It also removes carbon dioxide and other waste products from cells. Blood pressure helps mai
--------------------------------------------------------------------------------
[4/34] What are the ACC/AHA stages of heart failure?
The ACC/AHA st

In [17]:
finetuned_output_path = RESULTS_DIR / "finetuned_qwen_lora_answers.json"

with open(finetuned_output_path, "w", encoding="utf-8") as f:
    json.dump(finetuned_results, f, indent=2, ensure_ascii=False)

print("Saved fine-tuned answers to:", finetuned_output_path)

Saved fine-tuned answers to: d:\CardioBot_NLP_Final\results\finetuned_qwen_lora_answers.json
